In [ ]:
import pandas as pd
import numpy as np

previous_application = pd.read_csv(
    "data/raw/home_credit/previous_application.csv",
)

print(
    "previous_application shape:",
    previous_application.shape
)

display(previous_application.head())

In [ ]:
previous_application.info()

In [ ]:
previous_application.columns.tolist()

In [ ]:
print(
    "历史申请记录数量:",
    previous_application.shape[0]
)

print(
    "唯一历史申请数量:",
    previous_application[
        "SK_ID_PREV"
    ].nunique()
)

print(
    "唯一客户数量:",
    previous_application[
        "SK_ID_CURR"
    ].nunique()
)

In [ ]:
previous_application.duplicated().sum()

In [ ]:
previous_missing = (
    previous_application
    .isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
)

previous_missing["missing_rate"] = (
    previous_missing["missing_count"]
    / len(previous_application)
)

previous_missing.head(20)

In [ ]:
previous_application[
    "NAME_CONTRACT_STATUS"
].value_counts(
    dropna=False
)

In [ ]:
previous_application[
    "NAME_CONTRACT_STATUS"
].value_counts(
    normalize=True,
    dropna=False
)

In [ ]:
#建立特征工程1:历史申请状态辅助特征

In [ ]:
previous_application["IS_PREV_APPROVED"] = (
    previous_application[
        "NAME_CONTRACT_STATUS"
    ]
    .eq("Approved")
    .astype("int8")
)

previous_application["IS_PREV_REFUSED"] = (
    previous_application[
        "NAME_CONTRACT_STATUS"
    ]
    .eq("Refused")
    .astype("int8")
)

previous_application["IS_PREV_CANCELED"] = (
    previous_application[
        "NAME_CONTRACT_STATUS"
    ]
    .eq("Canceled")
    .astype("int8")
)

previous_application["IS_PREV_UNUSED"] = (
    previous_application[
        "NAME_CONTRACT_STATUS"
    ]
    .eq("Unused offer")
    .astype("int8")
)

In [ ]:
previous_application[
    [
        "SK_ID_CURR",
        "SK_ID_PREV",
        "NAME_CONTRACT_STATUS",
        "IS_PREV_APPROVED",
        "IS_PREV_REFUSED",
        "IS_PREV_CANCELED",
        "IS_PREV_UNUSED"
    ]
].head(10)

In [ ]:
previous_application[
    [
        "IS_PREV_APPROVED",
        "IS_PREV_REFUSED",
        "IS_PREV_CANCELED",
        "IS_PREV_UNUSED"
    ]
].sum()

In [ ]:
previous_application[
    "NAME_CONTRACT_STATUS"
].value_counts()

In [ ]:
#客户级历史申请数量与审批结果特征

In [ ]:
previous_status_features = (
    previous_application
    .groupby("SK_ID_CURR")
    .agg(
        PREV_APPLICATION_COUNT=(
            "SK_ID_PREV",
            "count"
        ),

        PREV_APPROVED_COUNT=(
            "IS_PREV_APPROVED",
            "sum"
        ),

        PREV_REFUSED_COUNT=(
            "IS_PREV_REFUSED",
            "sum"
        ),

        PREV_CANCELED_COUNT=(
            "IS_PREV_CANCELED",
            "sum"
        ),

        PREV_UNUSED_COUNT=(
            "IS_PREV_UNUSED",
            "sum"
        )
    )
    .reset_index()
)


previous_status_features[
    "PREV_APPROVED_RATE"
] = (
    previous_status_features[
        "PREV_APPROVED_COUNT"
    ]
    / previous_status_features[
        "PREV_APPLICATION_COUNT"
    ]
)

previous_status_features[
    "PREV_REFUSED_RATE"
] = (
    previous_status_features[
        "PREV_REFUSED_COUNT"
    ]
    / previous_status_features[
        "PREV_APPLICATION_COUNT"
    ]
)

previous_status_features[
    "PREV_CANCELED_RATE"
] = (
    previous_status_features[
        "PREV_CANCELED_COUNT"
    ]
    / previous_status_features[
        "PREV_APPLICATION_COUNT"
    ]
)

previous_status_features[
    "PREV_UNUSED_RATE"
] = (
    previous_status_features[
        "PREV_UNUSED_COUNT"
    ]
    / previous_status_features[
        "PREV_APPLICATION_COUNT"
    ]
)

previous_status_features.head()

In [ ]:
print(
    "previous_status_features shape:",
    previous_status_features.shape
)

print(
    "唯一客户数量:",
    previous_status_features[
        "SK_ID_CURR"
    ].nunique()
)

print(
    "重复客户数量:",
    previous_status_features[
        "SK_ID_CURR"
    ].duplicated().sum()
)

In [ ]:
status_rate_columns = [
    "PREV_APPROVED_RATE",
    "PREV_REFUSED_RATE",
    "PREV_CANCELED_RATE",
    "PREV_UNUSED_RATE"
]

previous_status_features[
    status_rate_columns
].describe().T

In [ ]:
previous_status_features[
    "STATUS_COUNT_CHECK"
] = (
    previous_status_features[
        "PREV_APPROVED_COUNT"
    ]
    + previous_status_features[
        "PREV_REFUSED_COUNT"
    ]
    + previous_status_features[
        "PREV_CANCELED_COUNT"
    ]
    + previous_status_features[
        "PREV_UNUSED_COUNT"
    ]
)

(
    previous_status_features[
        "STATUS_COUNT_CHECK"
    ]
    == previous_status_features[
        "PREV_APPLICATION_COUNT"
    ]
).value_counts()

In [ ]:
#客户级历史申请金额特征

In [ ]:
previous_amount_features = (
    previous_application
    .groupby("SK_ID_CURR")
    .agg(
        PREV_TOTAL_APPLICATION_AMOUNT=(
            "AMT_APPLICATION",
            "sum"
        ),

        PREV_AVG_APPLICATION_AMOUNT=(
            "AMT_APPLICATION",
            "mean"
        ),

        PREV_MAX_APPLICATION_AMOUNT=(
            "AMT_APPLICATION",
            "max"
        ),

        PREV_TOTAL_CREDIT_AMOUNT=(
            "AMT_CREDIT",
            "sum"
        ),

        PREV_AVG_CREDIT_AMOUNT=(
            "AMT_CREDIT",
            "mean"
        ),

        PREV_MAX_CREDIT_AMOUNT=(
            "AMT_CREDIT",
            "max"
        ),

        PREV_AVG_ANNUITY=(
            "AMT_ANNUITY",
            "mean"
        ),

        PREV_MAX_ANNUITY=(
            "AMT_ANNUITY",
            "max"
        ),

        PREV_AVG_DOWN_PAYMENT=(
            "AMT_DOWN_PAYMENT",
            "mean"
        ),

        PREV_MAX_DOWN_PAYMENT=(
            "AMT_DOWN_PAYMENT",
            "max"
        ),

        PREV_AVG_GOODS_PRICE=(
            "AMT_GOODS_PRICE",
            "mean"
        ),

        PREV_MAX_GOODS_PRICE=(
            "AMT_GOODS_PRICE",
            "max"
        )
    )
    .reset_index()
)

previous_amount_features.head()

In [ ]:
#申请金额与批准金额差异特征

In [ ]:
previous_application[
    "PREV_CREDIT_APPLICATION_DIFF"
] = (
    previous_application[
        "AMT_CREDIT"
    ]
    - previous_application[
        "AMT_APPLICATION"
    ]
)

previous_application[
    "PREV_CREDIT_APPLICATION_RATIO"
] = (
    previous_application[
        "AMT_CREDIT"
    ]
    / previous_application[
        "AMT_APPLICATION"
    ].replace(0, np.nan)
)

In [ ]:
previous_amount_gap_features = (
    previous_application
    .groupby("SK_ID_CURR")
    .agg(
        PREV_AVG_CREDIT_APPLICATION_DIFF=(
            "PREV_CREDIT_APPLICATION_DIFF",
            "mean"
        ),

        PREV_MIN_CREDIT_APPLICATION_DIFF=(
            "PREV_CREDIT_APPLICATION_DIFF",
            "min"
        ),

        PREV_MAX_CREDIT_APPLICATION_DIFF=(
            "PREV_CREDIT_APPLICATION_DIFF",
            "max"
        ),

        PREV_AVG_CREDIT_APPLICATION_RATIO=(
            "PREV_CREDIT_APPLICATION_RATIO",
            "mean"
        ),

        PREV_MIN_CREDIT_APPLICATION_RATIO=(
            "PREV_CREDIT_APPLICATION_RATIO",
            "min"
        ),

        PREV_MAX_CREDIT_APPLICATION_RATIO=(
            "PREV_CREDIT_APPLICATION_RATIO",
            "max"
        )
    )
    .reset_index()
)

previous_amount_gap_features.head()

In [ ]:
print(
    "previous_amount_features shape:",
    previous_amount_features.shape
)

print(
    "金额特征重复客户数量:",
    previous_amount_features[
        "SK_ID_CURR"
    ].duplicated().sum()
)

print(
    "previous_amount_gap_features shape:",
    previous_amount_gap_features.shape
)

print(
    "差异特征重复客户数量:",
    previous_amount_gap_features[
        "SK_ID_CURR"
    ].duplicated().sum()
)

In [ ]:
#客户级历史申请时间特征

In [ ]:
previous_time_features = (
    previous_application
    .groupby("SK_ID_CURR")
    .agg(
        PREV_LATEST_DECISION_DAYS=(
            "DAYS_DECISION",
            "max"
        ),

        PREV_EARLIEST_DECISION_DAYS=(
            "DAYS_DECISION",
            "min"
        ),

        PREV_AVG_DECISION_DAYS=(
            "DAYS_DECISION",
            "mean"
        )
    )
    .reset_index()
)


previous_time_features[
    "PREV_DECISION_HISTORY_SPAN"
] = (
    previous_time_features[
        "PREV_LATEST_DECISION_DAYS"
    ]
    - previous_time_features[
        "PREV_EARLIEST_DECISION_DAYS"
    ]
)


previous_time_features[
    "PREV_DAYS_SINCE_LATEST_APPLICATION"
] = (
    -previous_time_features[
        "PREV_LATEST_DECISION_DAYS"
    ]
)


previous_time_features[
    "PREV_DAYS_SINCE_EARLIEST_APPLICATION"
] = (
    -previous_time_features[
        "PREV_EARLIEST_DECISION_DAYS"
    ]
)


previous_time_features.head()

In [ ]:
print(
    "previous_time_features shape:",
    previous_time_features.shape
)

print(
    "唯一客户数量:",
    previous_time_features[
        "SK_ID_CURR"
    ].nunique()
)

print(
    "重复客户数量:",
    previous_time_features[
        "SK_ID_CURR"
    ].duplicated().sum()
)

In [ ]:
previous_time_features[
    [
        "PREV_LATEST_DECISION_DAYS",
        "PREV_EARLIEST_DECISION_DAYS",
        "PREV_DECISION_HISTORY_SPAN",
        "PREV_DAYS_SINCE_LATEST_APPLICATION",
        "PREV_DAYS_SINCE_EARLIEST_APPLICATION"
    ]
].describe().T

In [ ]:
print(
    "历史跨度小于 0 的客户数量:",
    (
        previous_time_features[
            "PREV_DECISION_HISTORY_SPAN"
        ] < 0
    ).sum()
)

print(
    "最近申请间隔小于 0 的客户数量:",
    (
        previous_time_features[
            "PREV_DAYS_SINCE_LATEST_APPLICATION"
        ] < 0
    ).sum()
)

In [ ]:
# 客户级贷款期限与还款压力特征

In [ ]:
previous_payment_features = (
    previous_application
    .groupby("SK_ID_CURR")
    .agg(
        PREV_AVG_PAYMENT_COUNT=(
            "CNT_PAYMENT",
            "mean"
        ),

        PREV_MAX_PAYMENT_COUNT=(
            "CNT_PAYMENT",
            "max"
        ),

        PREV_MIN_PAYMENT_COUNT=(
            "CNT_PAYMENT",
            "min"
        ),

        PREV_AVG_ANNUITY=(
            "AMT_ANNUITY",
            "mean"
        ),

        PREV_MAX_ANNUITY=(
            "AMT_ANNUITY",
            "max"
        ),

        PREV_AVG_CREDIT_AMOUNT=(
            "AMT_CREDIT",
            "mean"
        )
    )
    .reset_index()
)

In [ ]:
previous_payment_features[
    "PREV_AVG_ANNUITY_CREDIT_RATIO"
] = (
    previous_payment_features[
        "PREV_AVG_ANNUITY"
    ]
    / previous_payment_features[
        "PREV_AVG_CREDIT_AMOUNT"
    ].replace(0, np.nan)
)

previous_payment_features[
    "PREV_ESTIMATED_PAYMENT_TOTAL"
] = (
    previous_payment_features[
        "PREV_AVG_ANNUITY"
    ]
    * previous_payment_features[
        "PREV_AVG_PAYMENT_COUNT"
    ]
)

previous_payment_features.head()

In [ ]:
print(
    "previous_payment_features shape:",
    previous_payment_features.shape
)

print(
    "唯一客户数量:",
    previous_payment_features[
        "SK_ID_CURR"
    ].nunique()
)

print(
    "重复客户数量:",
    previous_payment_features[
        "SK_ID_CURR"
    ].duplicated().sum()
)

In [ ]:
#客户级历史申请类型特征

In [ ]:
previous_application[
    "NAME_CONTRACT_TYPE"
].value_counts(dropna=False)

In [ ]:
previous_application["IS_PREV_CASH_LOAN"] = (
    previous_application[
        "NAME_CONTRACT_TYPE"
    ]
    .eq("Cash loans")
    .astype("int8")
)

previous_application["IS_PREV_CONSUMER_LOAN"] = (
    previous_application[
        "NAME_CONTRACT_TYPE"
    ]
    .eq("Consumer loans")
    .astype("int8")
)

previous_application["IS_PREV_REVOLVING_LOAN"] = (
    previous_application[
        "NAME_CONTRACT_TYPE"
    ]
    .eq("Revolving loans")
    .astype("int8")
)

previous_application["IS_PREV_CONTRACT_UNKNOWN"] = (
    previous_application[
        "NAME_CONTRACT_TYPE"
    ]
    .eq("XNA")
    .astype("int8")
)

In [ ]:
previous_type_features = (
    previous_application
    .groupby("SK_ID_CURR")
    .agg(
        PREV_CASH_LOAN_COUNT=(
            "IS_PREV_CASH_LOAN",
            "sum"
        ),

        PREV_CONSUMER_LOAN_COUNT=(
            "IS_PREV_CONSUMER_LOAN",
            "sum"
        ),

        PREV_REVOLVING_LOAN_COUNT=(
            "IS_PREV_REVOLVING_LOAN",
            "sum"
        ),

        PREV_CONTRACT_UNKNOWN_COUNT=(
            "IS_PREV_CONTRACT_UNKNOWN",
            "sum"
        ),

        PREV_CONTRACT_TYPE_COUNT=(
            "NAME_CONTRACT_TYPE",
            "nunique"
        )
    )
    .reset_index()
)

In [ ]:
previous_type_features = (
    previous_type_features
    .merge(
        previous_status_features[
            [
                "SK_ID_CURR",
                "PREV_APPLICATION_COUNT"
            ]
        ],
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one"
    )
)

previous_type_features[
    "PREV_CASH_LOAN_RATE"
] = (
    previous_type_features[
        "PREV_CASH_LOAN_COUNT"
    ]
    / previous_type_features[
        "PREV_APPLICATION_COUNT"
    ]
)

previous_type_features[
    "PREV_CONSUMER_LOAN_RATE"
] = (
    previous_type_features[
        "PREV_CONSUMER_LOAN_COUNT"
    ]
    / previous_type_features[
        "PREV_APPLICATION_COUNT"
    ]
)

previous_type_features[
    "PREV_REVOLVING_LOAN_RATE"
] = (
    previous_type_features[
        "PREV_REVOLVING_LOAN_COUNT"
    ]
    / previous_type_features[
        "PREV_APPLICATION_COUNT"
    ]
)

previous_type_features[
    "PREV_CONTRACT_UNKNOWN_RATE"
] = (
    previous_type_features[
        "PREV_CONTRACT_UNKNOWN_COUNT"
    ]
    / previous_type_features[
        "PREV_APPLICATION_COUNT"
    ]
)

previous_type_features.head()

In [ ]:
print(
    "previous_type_features shape:",
    previous_type_features.shape
)

print(
    "唯一客户数量:",
    previous_type_features[
        "SK_ID_CURR"
    ].nunique()
)

print(
    "重复客户数量:",
    previous_type_features[
        "SK_ID_CURR"
    ].duplicated().sum()
)

In [ ]:
#客户级首付款与贷款条件特征

In [ ]:
# 1. 建立申请级贷款覆盖比例
previous_application[
    "PREV_CREDIT_GOODS_RATIO"
] = (
    previous_application["AMT_CREDIT"]
    / previous_application["AMT_GOODS_PRICE"].replace(0, np.nan)
)


# 2. 标记该笔申请是否有有效的首付款信息
previous_application[
    "HAS_DOWN_PAYMENT_INFO"
] = (
    previous_application["AMT_DOWN_PAYMENT"]
    .notna()
    .astype("int8")
)


# 3. 标记该笔申请是否明确为零首付款
previous_application[
    "IS_PREV_ZERO_DOWN_PAYMENT"
] = (
    previous_application["AMT_DOWN_PAYMENT"]
    .eq(0)
    .astype("int8")
)


# 4. 聚合成客户级特征
previous_condition_features = (
    previous_application
    .groupby("SK_ID_CURR")
    .agg(
        PREV_AVG_DOWN_PAYMENT_RATE=(
            "RATE_DOWN_PAYMENT",
            "mean"
        ),

        PREV_MAX_DOWN_PAYMENT_RATE=(
            "RATE_DOWN_PAYMENT",
            "max"
        ),

        PREV_AVG_CREDIT_GOODS_RATIO=(
            "PREV_CREDIT_GOODS_RATIO",
            "mean"
        ),

        PREV_MAX_CREDIT_GOODS_RATIO=(
            "PREV_CREDIT_GOODS_RATIO",
            "max"
        ),

        PREV_ZERO_DOWN_PAYMENT_COUNT=(
            "IS_PREV_ZERO_DOWN_PAYMENT",
            "sum"
        ),

        PREV_DOWN_PAYMENT_INFO_COUNT=(
            "HAS_DOWN_PAYMENT_INFO",
            "sum"
        )
    )
    .reset_index()
)


# 5. 建立零首付款比例
previous_condition_features[
    "PREV_ZERO_DOWN_PAYMENT_RATE"
] = (
    previous_condition_features[
        "PREV_ZERO_DOWN_PAYMENT_COUNT"
    ]
    / previous_condition_features[
        "PREV_DOWN_PAYMENT_INFO_COUNT"
    ].replace(0, np.nan)
)


previous_condition_features.head()

In [ ]:
#合并前面创建的特征表

In [ ]:
# 1. 删除 previous_payment_features 中已经重复建立的金额特征
previous_payment_features = (
    previous_payment_features
    .drop(
        columns=[
            "PREV_AVG_CREDIT_AMOUNT",
            "PREV_AVG_ANNUITY",
            "PREV_MAX_ANNUITY"
        ]
    )
)


# 2. 查看清理后的字段
previous_payment_features.columns.tolist()

In [ ]:
# 1. 删除只用于计算比例的重复申请次数字段
previous_type_features = (
    previous_type_features
    .drop(
        columns=[
            "PREV_APPLICATION_COUNT"
        ]
    )
)


# 2. 查看清理后的字段
previous_type_features.columns.tolist()

In [ ]:
# 1. 以历史申请状态特征作为客户级主表
previous_application_features = (
    previous_status_features

    # 2. 合并历史申请金额特征
    .merge(
        previous_amount_features,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one"
    )

    # 3. 合并申请金额与批准金额差异特征
    .merge(
        previous_amount_gap_features,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one"
    )

    # 4. 合并历史申请时间特征
    .merge(
        previous_time_features,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one"
    )

    # 5. 合并贷款期限与还款压力特征
    .merge(
        previous_payment_features,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one"
    )

    # 6. 合并历史申请类型特征
    .merge(
        previous_type_features,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one"
    )

    # 7. 合并首付款与贷款覆盖特征
    .merge(
        previous_condition_features,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one"
    )
)


# 8. 查看结果
previous_application_features.head()

In [ ]:
duplicate_suffix_columns = [
    col
    for col in previous_application_features.columns
    if col.endswith("_x") or col.endswith("_y")
]

duplicate_suffix_columns

In [ ]:
print(
    "申请总次数字段是否存在:",
    "PREV_APPLICATION_COUNT"
    in previous_application_features.columns
)

print(
    "平均批准金额字段是否存在:",
    "PREV_AVG_CREDIT_AMOUNT"
    in previous_application_features.columns
)

print(
    "平均还款金额字段是否存在:",
    "PREV_AVG_ANNUITY"
    in previous_application_features.columns
)

In [ ]:
# 1. 检查最终数据规模
print(
    "previous_application_features shape:",
    previous_application_features.shape
)

print(
    "原始客户数量:",
    previous_application["SK_ID_CURR"].nunique()
)


# 2. 检查是否仍然是一行一个客户
print(
    "唯一客户数量:",
    previous_application_features[
        "SK_ID_CURR"
    ].nunique()
)

print(
    "重复客户数量:",
    previous_application_features[
        "SK_ID_CURR"
    ].duplicated().sum()
)


# 3. 检查是否还有 _x 或 _y 字段
duplicate_suffix_columns = [
    col
    for col in previous_application_features.columns
    if col.endswith("_x") or col.endswith("_y")
]

print(
    "重复后缀字段:",
    duplicate_suffix_columns
)

In [ ]:
#针对已经合并后的最终表进行检查

In [ ]:
# 1. 统计每个字段的缺失数量
previous_missing_summary = (
    previous_application_features
    .isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
)


# 2. 计算缺失比例
previous_missing_summary[
    "missing_rate"
] = (
    previous_missing_summary[
        "missing_count"
    ]
    / len(previous_application_features)
)


# 3. 查看缺失最多的字段
previous_missing_summary.head(20)

In [ ]:
# 1. 找出所有比例类字段
previous_ratio_columns = [
    col
    for col in previous_application_features.columns
    if "RATE" in col or "RATIO" in col
]


# 2. 查看比例字段分布
previous_application_features[
    previous_ratio_columns
].describe().T

In [ ]:
previous_application_features[
    [
        "PREV_APPLICATION_COUNT",
        "PREV_TOTAL_APPLICATION_AMOUNT",
        "PREV_TOTAL_CREDIT_AMOUNT",
        "PREV_AVG_PAYMENT_COUNT",
        "PREV_MAX_PAYMENT_COUNT",
        "PREV_AVG_ANNUITY_CREDIT_RATIO",
        "PREV_ESTIMATED_PAYMENT_TOTAL"
    ]
].describe().T

In [ ]:
# 1. 保存客户级特征表
previous_application_features.to_parquet(
    "data/processed/previous_application_features.parquet",
    index=False
)

In [ ]:
from pathlib import Path


# 2. 建立输出路径
output_path = Path(
    "data/processed/previous_application_features.parquet"
)


# 3. 检查文件是否成功保存
print(
    "文件是否存在:",
    output_path.exists()
)

print(
    "保存位置:",
    output_path.resolve()
)

print(
    "最终 Shape:",
    previous_application_features.shape
)

In [ ]:
# 1. 查看缺失值
previous_application_features.isna().sum().sort_values(
    ascending=False
).head(20)


# 2. 保存最终客户级特征表
previous_application_features.to_parquet(
    "data/processed/previous_application_features.parquet",
    index=False
)